In [ ]:
from matplotlib import pyplot as plt
import torch
from sklearn.metrics import roc_curve, f1_score, auc
from tools.embs_tools import get_embs, accelerated_cosine_similarity
import numpy as np

In [ ]:
all_embs = get_embs("/home/simon/car_outputs/")

In [ ]:
path = list(all_embs.keys())[3]
avg_fn = lambda embs: np.mean(embs, axis=0)
embs = torch.stack([torch.tensor(emb) for _, _, emb in all_embs[path]])
ids = torch.tensor([id for id, _, _ in all_embs[path]])

similarity = accelerated_cosine_similarity(embs, embs, batch_size=2048).flatten()
issame_tracks = (ids.unsqueeze(0) == ids.unsqueeze(1)).flatten()

In [ ]:
len(similarity), len(issame_tracks)

In [ ]:
threshold = 0.5
matches = similarity > threshold
issame_tracks = issame_tracks > 0
f1 = f1_score(issame_tracks, matches)

print(f"F1 Score: {f1}")
print(f"Accuracy: {(matches == issame_tracks).sum() / len(matches)}")

In [ ]:
fpr, tpr, threshold = roc_curve(issame_tracks, similarity)
print(f"ROC AUC: {auc(fpr, tpr)}")

In [ ]:
issame_tracks = issame_tracks.to("cuda")
similarity = similarity.to("cuda")

same_sims = similarity[issame_tracks].cpu().numpy()
diff_sims = similarity[~issame_tracks].cpu().numpy()

# plot violin plots for same and different tracks
plt.boxplot([same_sims, diff_sims], showmeans=True, showfliers=False)
plt.xticks([1, 2], ['Same Tracks', 'Different Tracks'])
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity Distribution for Same and Different Tracks')

In [ ]:
range = 0.9

plt.xlim([range - 1.0, range])
plt.ylim([1.0 - range, 2.0 - range])
plt.plot(fpr, tpr)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.grid(True)
plt.show()

In [ ]:
plt.show()
plt.figure(figsize=(8, 6))
plt.plot(threshold, tpr, label='True Positive Rate (TPR)')
plt.plot(threshold, fpr, label='False Positive Rate (FPR)')
plt.plot(threshold, tpr - fpr, label='TPR - FPR', linestyle='--')
plt.xlim([0.0, 1.0])
plt.xlabel('Threshold')
plt.ylabel('Rate')
plt.title('TPR and FPR vs. Threshold')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
J = tpr - fpr
ix = np.argmax(J)
best_threshold = threshold[ix]
print(f'Best Threshold: {best_threshold}')